## Notebook for testing gridded stat and anomaly methods

### Import libraries and initiate utilities routines

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import xarray as xr
import os
from datetime import datetime
import matplotlib.pyplot as plt
import geopandas as gpd

current_path = Path.cwd()

# Search for the utilities directory
for parent in current_path.parents:
    candidate = parent / "utilities"
    if candidate.is_dir():
        sys.path.insert(0, str(parent))
        break
else:
    raise FileNotFoundError("Could not locate 'utilities' directory in parent paths.")

from utilities import init_notebook_environment
from utilities import get_dataset_products,get_prod_files,dataset_defaults,run_psc_pipeline,subset_dataset, parse_dataset_info,get_dates,get_source_file_dates
from utilities import period_info,resolve_dataset_map,get_dataset_dirs,get_nc_prod,product_defaults,netcdf_product_defaults
from utilities import get_pyfile_functions,find_function, find_text,build_product_attributes
from utilities import run_stats_pipeline,build_stats_map, parse_dataset_info, file_parser
from utilities import date_utilities, get_dates,get_source_file_dates,get_file_dates,get_default_dataset_product
from utilities import period_utilities, period_info, get_period_info, get_period_sets, get_period_dates, extract_period_code, check_period, generate_periods, date_to_daily_period


env = init_notebook_environment(verbose=False)


## Definitions
**Dataset** - The source of the original data (e.g. GLORYS, OC-CCI, MOM6, ACSPO)  
**Version** - The dataset version (e.g. V6.0)  
**Prod** - Specific product(s) within datasets  
**Period** - The temporal resolution (e.g. daily, weekly, monthly, seasonal, annual)  
**Climatology Period** - The temporal resolution on the climatology (DOY, WEEK, MONTH, SEASON, ANNUAL)  
**Subset** - A predefined area of interest to subset the global data 
* NES - Northeast Shelf [lat_min:34.0, lat_max:47.0, lon_min:-79.0, lon_max:-61.0 ]
* NWA - Northwest Atlantic (i.e. East Coast) [lat_min:22.5, lat_max:48.5, lon_min:-82.5, lon_max:-51.5]  


## Workflow
<img src="../StatsWorkflows-20260430.png">


## Stats Pipeline
Headless Command Line Interface (CLI) script that can be incorporated into the automated workflows
* Adds the script path & loads the utility scripts
* Parses the input arguments and initiates the Pipeline

    ### Run Stats Pipeline
    Runs the multi-product, multi-period stats pipeline either sequential or in parallel

    ### Build Stats Map
    * Finds the input files
    * Generate the period map
    * Create the output directory & file names
    * Compare inputs and outputs to determine if the file exists and/or needs to be recreated

        ### Get Prod Files
        Retrieves NetCDF files for a specified product from a structured dataset directory (defaults to daily ).

### Get Product Defaults 
* "EDAB" product defaults
* netcdf (i.e. source data) product defaults

In [2]:
product_defaults()

{'CHL': ('CHL', 'OCCCI', 'SOURCE'),
 'CHLOR_A': ('CHLOR_A', 'OCCCI', 'PRODUCT'),
 'SST': ('SST', 'ACSPO', 'SOURCE'),
 'PPD': ('PPD', 'OCCCI', 'PRODUCT'),
 'PSC': ('PSC', 'OCCCI', 'PRODUCT'),
 'RRS': ('RRS', 'OCCCI', 'SOURCE'),
 'PAR': ('PAR', 'GLOBCOLOUR', 'SOURCE'),
 'IPAR': ('IPAR', 'PACE', 'SOURCE'),
 'AVW': ('AVW', 'OCCCI', 'PRODUCT'),
 'KD': ('KD', 'OCCCI', 'SOURCE'),
 'IOP': ('IOP', 'OCCCI', 'SOURCE'),
 'MOANA': ('MOANA', 'PACE', 'SOURCE'),
 'CARBON': ('CARBON', 'PACE', 'SOURCE'),
 'FLH': ('FLH', 'PACE', 'SOURCE'),
 'CHL_TEMP': ('CHL', 'GLOBCOLOUR', 'SOURCE'),
 'SST_TEMP': ('SST', 'ACSPONRT', 'SOURCE'),
 'CHL_FRONTS': ('CHL_FRONTS', 'OCCCI', 'PRODUCT'),
 'SST_FRONTS': ('SST_FRONTS', 'ACSPO', 'PRODUCT'),
 'FRONTS': ('SST_FRONTS', 'ACSPO', 'PRODUCT'),
 'BTEMP': ('BTEMP', 'GLORYS', 'SOURCE')}

In [2]:
netcdf_product_defaults()

{'ACSPO': {'SST': 'sea_surface_temperature',
  'SST_GRADMAG': 'sst_gradient_magnitude',
  'SST_GRADDIR': 'sst_gradient_direction',
  'SST_BIAS': 'sses_bias'},
 'ACSPO_NRT': {'SST': 'sea_surface_temperature',
  'SST_GRADMAG': 'sst_gradient_magnitude',
  'SST_GRADDIR': 'sst_gradient_direction',
  'SST_BIAS': 'sses_bias'},
 'CORALSST': {'SST': 'analysed_sst'},
 'OISST': {'SST': 'SST'},
 'MUR': {'SST': 'analysed_sst'},
 'AVHRR': {'SST': 'sea_surface_temperature'},
 'GLOBCOLOUR': {'PAR': 'PAR_mean', 'CHL': 'CHL1_mean'},
 'OCCCI': {'CHL': 'chlor_a'},
 'PACE': {'CHL': 'chlor_a',
  'PAR': 'par_day_planar_above',
  'IPAR': 'ipar_plana_above',
  'RRS': 'Rrs',
  'AVW': 'avw',
  'MOANA_PRO': 'prococcus_moana',
  'MOANA_SYN': 'synococcus_moana',
  'MOANA_PICO': 'picoeuk_moana',
  'PYTHO_CARBON': 'carbon_phyto',
  'FLH': 'nflh'},
 'TESTDATASET': {'SST_MEAN': 'sst_mean', 'SST_MAX': 'sst_max'}}

### Get Product files

In [4]:
get_prod_files('CHL')

📦 Found 454 .nc files in: /Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/SOURCE/GLOBAL_4KM_DAILY/CHL


['/Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/SOURCE/GLOBAL_4KM_DAILY/CHL/ESACCI-OC-L3S-CHLOR_A-MERGED-1D_DAILY_4km_GEO_PML_OCx-19980101-fv6.0.nc',
 '/Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/SOURCE/GLOBAL_4KM_DAILY/CHL/ESACCI-OC-L3S-CHLOR_A-MERGED-1D_DAILY_4km_GEO_PML_OCx-19980102-fv6.0.nc',
 '/Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/SOURCE/GLOBAL_4KM_DAILY/CHL/ESACCI-OC-L3S-CHLOR_A-MERGED-1D_DAILY_4km_GEO_PML_OCx-19980103-fv6.0.nc',
 '/Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/SOURCE/GLOBAL_4KM_DAILY/CHL/ESACCI-OC-L3S-CHLOR_A-MERGED-1D_DAILY_4km_GEO_PML_OCx-19980104-fv6.0.nc',
 '/Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/SOURCE/GLOBAL_4KM_DAILY/CHL/ESACCI-OC-L3S-CHLOR_A-MERGED-1D_DAILY_4km_GEO_PML_OCx-19980105-fv6.0.nc',
 '/Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/SOURCE/GLOBAL_4KM_DAILY/CHL/ESACCI-OC-L3S-CHLOR_A-MERGED-1D_DAILY_4km_GEO_PML_OCx-19980106-fv6.0.nc',
 '/Users/kimberly.hyde/Docum

In [14]:
get_prod_files('CHL',dataset='GLOBCOLOUR')

📦 Found 1 .nc files in: /Users/kimberly.hyde/Documents/nadata/DATASETS/GLOBCOLOUR/V4.2.1/SOURCE/BINNED_4KM_DAILY/CHL


['/Users/kimberly.hyde/Documents/nadata/DATASETS/GLOBCOLOUR/V4.2.1/SOURCE/BINNED_4KM_DAILY/CHL/L3b_20250909__GLOB_4_GSM-MODVIR_CHL1_DAY_00.nc']

In [5]:
get_prod_files('BTEMP')

⚠ No .nc files found in: /Users/kimberly.hyde/Documents/nadata/DATASETS/GLORYS/V12/SOURCE/MAPPED_9KM_DAILY/BTEMP


[]

In [16]:
get_prod_files('CHL',period='M',map_region='NES')

📦 Found 22 .nc files in: /Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/PRODUCTS/NES_4KM_MONTHLY/CHL


['/Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/PRODUCTS/NES_4KM_MONTHLY/CHL/M_199801-OCCCI-CHL-NES-STATS.nc',
 '/Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/PRODUCTS/NES_4KM_MONTHLY/CHL/M_199802-OCCCI-CHL-NES-STATS.nc',
 '/Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/PRODUCTS/NES_4KM_MONTHLY/CHL/M_199803-OCCCI-CHL-NES-STATS.nc',
 '/Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/PRODUCTS/NES_4KM_MONTHLY/CHL/M_199804-OCCCI-CHL-NES-STATS.nc',
 '/Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/PRODUCTS/NES_4KM_MONTHLY/CHL/M_199805-OCCCI-CHL-NES-STATS.nc',
 '/Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/PRODUCTS/NES_4KM_MONTHLY/CHL/M_199806-OCCCI-CHL-NES-STATS.nc',
 '/Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/PRODUCTS/NES_4KM_MONTHLY/CHL/M_199807-OCCCI-CHL-NES-STATS.nc',
 '/Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/PRODUCTS/NES_4KM_MONTHLY/CHL/M_199808-OCCCI-CHL-NES-STATS.nc',
 '/Users/kimberl

In [17]:
get_prod_files('CHL',getfilepath=True,period='MONTH',map_region='NES')

⚠ Warning: Resolved path does not exist on disk: /Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/PRODUCTS/NES_4KM_CLIMATOLOGY/CHL
  (Hint: Set make_dir=True to automatically create this directory structure)


'/Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/PRODUCTS/NES_4KM_CLIMATOLOGY/CHL'

## Stat Periods
<img src="../StatsPeriods-20260430.png">

In [3]:
info = get_period_info()

print("Available period codes:")
for p, meta in info.items():
    file_format = meta["file_format"]
    description = meta["description"]
    print(f"{p:5}  {file_format:30}  {description}")

Available period codes:
D      D_YYYYMMDD                      Daily
DD     DD_YYYYMMDD_YYYYMMDD            Range of daily data
DOY    DOY_DDD_YYYY_YYYY               Climatological day of year
DOYS   DOYS_YYYY_YYYY                  Combined climatological days of year
D3     D3_YYYYMMDD_YYYYMMDD            Running 3-day
DD3    DD3_YYYYMMDD_YYYYMMDD           Range of running 3-day data
D8     D8_YYYYMMDD_YYYYMMDD            Running 8-day
DD8    DD8_YYYYMMDD_YYYYMMDD           Range of running 8-day data
W      W_YYYYWW                        Weekly (ISO week format)
WW     WW_YYYYWW_YYYYWW                Range of weekly data
WEEK   WEEK_WW_YYYY_YYYY               Climatological ISO week
WEEKS  WEEKS_YYYY_YYYY                 Combined climatological ISO week
M      M_YYYYMM                        Monthly
MM     MM_YYYYMM_YYYYMM                Range of monthly data
MONTH  MONTH_MM_YYYY_YYYY              Climatological month
MONTHS  MONTHS_YYYY_YYYY                Combined climatological

In [5]:
get_period_info('W')

{'groupby': 'week',
 'input_period_code': 'D',
 'is_range': False,
 'is_running_mean': False,
 'is_climatology': False,
 'iso_duration': 'P1W',
 'groupby_hint': 'time.dt.isocalendar().week',
 'file_format': 'W_YYYYWW',
 'date_format': 'YYYYWW',
 'n_date_segments': 1,
 'description': 'Weekly (ISO week format)',
 'folder_name': 'Weekly',
 'regex': re.compile(r'^W_(?P<seg0>\d{6})$', re.UNICODE),
 'file_format_parts': ['YYYYWW']}

#### Weekly periods
Weekly period are based on the standardized ISO week format. Week 1 of the 7-day, Monday-to-Sunday, week-based calendar is defined as the week containing the first Thursday of the year, usually containing January 4th. Thus week one of the new calendar year may start in December of the preceeding year and week 52/53 may end in January of the following year.
*  W → a SINGLE ISO week (YYYYWW) 
*  WW → a RANGE of ISO weeks (YYYYWW_YYYYWW)
*  WEEK → a SINGLE CLIMATOLOGY of a week with the year span (WW_YYYY_YYYY)
*  WEEKS → a COMPLETE CLIMATOLOGY series (week 1 to 52) with the year span (WW_YYYY_YYYY)

In [9]:
pers = [
    "W_201001",
    "W_201101",
    "W_201201",
    "W_201301",
    "W_201401",
    "WW_202101_202152",
    "WEEK_03_2006_2020",
    "WEEKS_2000_2020"
    ]
pds = get_period_dates(pers)

for per, pd in zip(pers, pds):
    print(f"{per} : {pd}")

W_201001 : ('20100104', '20100110')
W_201101 : ('20110103', '20110109')
W_201201 : ('20120102', '20120108')
W_201301 : ('20121231', '20130106')
W_201401 : ('20131230', '20140105')
WW_202101_202152 : ('20210104', '20220102')
WEEK_03_2006_2020 : ('20060116', '20200119')
WEEKS_2000_2020 : ('20000101', '20201231')


In [4]:
info = get_period_info()
periods = info.keys()
for p in periods:
    i = info.get(p)
    print(f"{p} is climatology = {i['is_climatology']}")

D is climatology = False
DD is climatology = False
DOY is climatology = True
DOYS is climatology = True
D3 is climatology = False
DD3 is climatology = False
D8 is climatology = False
DD8 is climatology = False
W is climatology = False
WW is climatology = False
WEEK is climatology = True
WEEKS is climatology = True
M is climatology = False
MM is climatology = False
MONTH is climatology = True
MONTHS is climatology = True
M3 is climatology = False
MM3 is climatology = False
JFM is climatology = False
AMJ is climatology = False
JAS is climatology = False
OND is climatology = False
SEA is climatology = False
SJFM is climatology = True
SAMJ is climatology = True
SJAS is climatology = True
SOND is climatology = True
SEASON is climatology = True
A is climatology = False
AA is climatology = False
ANNUAL is climatology = True
Y is climatology = False
YY is climatology = False
YEAR is climatology = True


### Create periods from a given date range

In [ ]:
start = "19970901"
end = "20260309"
climatology_range = [1991,2020]

d = generate_periods('D',start,end)
d8 = generate_periods('D8',start,end)
dd8y = generate_periods('DD8',start,end,range_by_year=True)
doy = generate_periods('DOY',start,end,climatology_range=climatology_range)
doys = generate_periods('DOYS',start,end,climatology_range=climatology_range)
w = generate_periods('W',start,end)
m = generate_periods('M',start,end)
jfm = generate_periods('JFM',start,end)
annual = generate_periods('ANNUAL',start,end,climatology_range=climatology_range)

print(f"Daily periods: {d[:5]} ... {d[-5:]}")
print(f"8-day periods: {d8[:5]} ... {d8[-5:]}")
print(f"8-day grouped by yearperiods: {dd8y}")
print(f"Daily Climatology: {doy[:5]} ... {doy[-5:]}")
print(f"DOY Climatology: {doys}")
print(f"Weekly periods: {w[:5]} ... {w[-5:]}")
print(f"Monthly periods: {m[:5]} ... {m[-5:]}")
print(f"JFM periods: {jfm[:5]} ... {jfm[-5:]}")
print(f"Annual Climatology: {annual}")

Daily periods: ['D_19970901', 'D_19970902', 'D_19970903', 'D_19970904', 'D_19970905'] ... ['D_20260305', 'D_20260306', 'D_20260307', 'D_20260308', 'D_20260309']
8-day periods: ['D8_19970901_19970908', 'D8_19970902_19970909', 'D8_19970903_19970910', 'D8_19970904_19970911', 'D8_19970905_19970912'] ... ['D8_20260305_20260312', 'D8_20260306_20260313', 'D8_20260307_20260314', 'D8_20260308_20260315', 'D8_20260309_20260316']
8-day grouped by yearperiods: ['DD8_19970901_19980107', 'DD8_19980101_19990107', 'DD8_19990101_20000107', 'DD8_20000101_20010107', 'DD8_20010101_20020107', 'DD8_20020101_20030107', 'DD8_20030101_20040107', 'DD8_20040101_20050107', 'DD8_20050101_20060107', 'DD8_20060101_20070107', 'DD8_20070101_20080107', 'DD8_20080101_20090107', 'DD8_20090101_20100107', 'DD8_20100101_20110107', 'DD8_20110101_20120107', 'DD8_20120101_20130107', 'DD8_20130101_20140107', 'DD8_20140101_20150107', 'DD8_20150101_20160107', 'DD8_20160101_20170107', 'DD8_20170101_20180107', 'DD8_20180101_20190107

### Create sets of periods based on the input period "files" from above

In [21]:
doy_perset = get_period_sets('DOY',files=d)
print(f"DOY period sets: {list(doy_perset.keys())}")
doy_perset

DOY period sets: ['target_code', 'expected_input_code', 'start_dt', 'end_dt', 'period_map', 'diagnostics']


{'target_code': 'DOY',
 'expected_input_code': 'D',
 'start_dt': '19970101',
 'end_dt': '20201231',
 'period_map': {'DOY_001_1997_2020': {'input_periods': ['D_19980101',
    'D_19990101',
    'D_20000101',
    'D_20010101',
    'D_20020101',
    'D_20030101',
    'D_20040101',
    'D_20050101',
    'D_20060101',
    'D_20070101',
    'D_20080101',
    'D_20090101',
    'D_20100101',
    'D_20110101',
    'D_20120101',
    'D_20130101',
    'D_20140101',
    'D_20150101',
    'D_20160101',
    'D_20170101',
    'D_20180101',
    'D_20190101',
    'D_20200101'],
   'input_files': ['D_19980101',
    'D_19990101',
    'D_20000101',
    'D_20010101',
    'D_20020101',
    'D_20030101',
    'D_20040101',
    'D_20050101',
    'D_20060101',
    'D_20070101',
    'D_20080101',
    'D_20090101',
    'D_20100101',
    'D_20110101',
    'D_20120101',
    'D_20130101',
    'D_20140101',
    'D_20150101',
    'D_20160101',
    'D_20170101',
    'D_20180101',
    'D_20190101',
    'D_20200101']},
  

### Build Stats Map
<img src="../BuildStatsMap-20260430.png">

In [2]:
build_stats_map('CHL','M',subset='NES')


🔍 DEBUG: Output directory for CHL (M) is /Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/PRODUCTS/NES_4KM_MONTHLY/CHL
📦 Found 454 .nc files in: /Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/SOURCE/GLOBAL_4KM_DAILY/CHL
------------------------------------------------------------
📊 STATS SUMMARY: CHL | M | NES
📁 Output Dir: /Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/PRODUCTS/NES_4KM_MONTHLY/CHL
✅ Up-to-date:  0/85 (0.0%)
⏳ To process:  85 files
------------------------------------------------------------


{'M_199801': {'inputs': ['/Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/SOURCE/GLOBAL_4KM_DAILY/CHL/ESACCI-OC-L3S-CHLOR_A-MERGED-1D_DAILY_4km_GEO_PML_OCx-19980101-fv6.0.nc',
   '/Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/SOURCE/GLOBAL_4KM_DAILY/CHL/ESACCI-OC-L3S-CHLOR_A-MERGED-1D_DAILY_4km_GEO_PML_OCx-19980102-fv6.0.nc',
   '/Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/SOURCE/GLOBAL_4KM_DAILY/CHL/ESACCI-OC-L3S-CHLOR_A-MERGED-1D_DAILY_4km_GEO_PML_OCx-19980103-fv6.0.nc',
   '/Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/SOURCE/GLOBAL_4KM_DAILY/CHL/ESACCI-OC-L3S-CHLOR_A-MERGED-1D_DAILY_4km_GEO_PML_OCx-19980104-fv6.0.nc',
   '/Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/SOURCE/GLOBAL_4KM_DAILY/CHL/ESACCI-OC-L3S-CHLOR_A-MERGED-1D_DAILY_4km_GEO_PML_OCx-19980105-fv6.0.nc',
   '/Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/SOURCE/GLOBAL_4KM_DAILY/CHL/ESACCI-OC-L3S-CHLOR_A-MERGED-1D_DAILY_4km_GEO_PML_OCx-19980106-fv6.0

#### Process output files in parallel using ProcessPoolExecutor
<img src="../RunStatsPipeline-20260430.png">
    
        with concurrent.futures.ProcessPoolExecutor(parallel_runs=parallel_runs) as executor:
            # Submit all tasks to the pool
            future_to_period = {
                executor.submit(process_single_stat, task, prod, period, verbose, **kwargs): out_period
                for out_period, task in tasks_to_run
            }
        
        # Collect results as they finish (not necessarily in order!)
        for future in concurrent.futures.as_completed(future_to_period):
            out_period = future_to_period[future]
            try:
                success, corrupt_files = future.result()
                if corrupt_files: all_corrupt_inputs.extend(corrupt_files)
                
                if success:
                    successful_tasks.append(out_period)
                    if verbose: print(f"  ✅ [Parallel] Finished {out_period}")
                else:
                    failed_tasks.append(out_period)
            except Exception as e:
                failed_tasks.append(f"{out_period}: {str(e)}")
                print(f"\n❌ [Parallel] Error on {out_period}:\n  ⚠️  {e}")


### Process Single Stat
<img src="../ProcessSingleStat-20260430.png">

### Compute Stats
Default = ['mean', 'min', 'max', 'median', 'std', 'sum', 'var', 'count']

### Add Attributes
* Global
* Product

In [3]:
xr.open_mfdataset(get_prod_files('CHL',period='M',map_region='NES'))

📦 Found 22 .nc files in: /Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/PRODUCTS/NES_4KM_MONTHLY/CHL


<xarray.Dataset> Size: 107MB
Dimensions:     (time: 22, lat: 312, lon: 432)
Coordinates:
  * lat         (lat) float64 2kB 46.98 46.94 46.9 46.85 ... 34.1 34.06 34.02
  * lon         (lon) float64 3kB -78.98 -78.94 -78.9 ... -61.1 -61.06 -61.02
  * time        (time) datetime64[ns] 176B 1998-01-01 1998-02-01 ... 2005-01-01
Data variables:
    CHL_mean    (time, lat, lon) float32 12MB dask.array<chunksize=(1, 312, 432), meta=np.ndarray>
    CHL_min     (time, lat, lon) float32 12MB dask.array<chunksize=(1, 312, 432), meta=np.ndarray>
    CHL_max     (time, lat, lon) float32 12MB dask.array<chunksize=(1, 312, 432), meta=np.ndarray>
    CHL_median  (time, lat, lon) float32 12MB dask.array<chunksize=(1, 312, 432), meta=np.ndarray>
    CHL_std     (time, lat, lon) float32 12MB dask.array<chunksize=(1, 312, 432), meta=np.ndarray>
    CHL_sum     (time, lat, lon) float32 12MB dask.array<chunksize=(1, 312, 432), meta=np.ndarray>
    CHL_var     (time, lat, lon) float32 12MB dask.array<chunksize=(1, 312, 432), meta=np.ndarray>
    CHL_count   (time, lat, lon) float64 24MB dask.array<chunksize=(1, 312, 432), meta=np.ndarray>
Attributes: (12/46)
    conventions:                   CF-1.11, COARDS, ACDD-1.3
    acknowledgement:               The data are sponsored by NOAA and may be ...
    institution:                   DOC | NOAA | National Marine Fisheries Ser...
    naming_authority:              gov.noaa.nefsc
    license:                       The data may be used and redistributed for...
    creator_type:                  group
    ...                            ...
    source_dataset_url:            https://climate.esa.int/en/projects/ocean-...
    source_dataset_reference:      Sathyendranath, S, Brewin, RJW, Brockmann,...
    source_version_url:            https://catalogue.ceda.ac.uk/uuid/9aba59ec...
    source_version_doi:            doi:10.5285/5011d22aae5a4671b0cbc7d05c56c4f0
    source_version_reference:      Sathyendranath, S.; Jackson, T.; Brockmann...
    source_version:                V6.0

In [4]:
ds = xr.open_mfdataset(get_prod_files('CHL',period='M',map_region='NES'))
da = ds.CHL_mean
da

📦 Found 22 .nc files in: /Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/PRODUCTS/NES_4KM_MONTHLY/CHL


<xarray.DataArray 'CHL_mean' (time: 22, lat: 312, lon: 432)> Size: 12MB
dask.array<concatenate, shape=(22, 312, 432), dtype=float32, chunksize=(1, 312, 432), chunktype=numpy.ndarray>
Coordinates:
  * lat      (lat) float64 2kB 46.98 46.94 46.9 46.85 ... 34.15 34.1 34.06 34.02
  * lon      (lon) float64 3kB -78.98 -78.94 -78.9 ... -61.1 -61.06 -61.02
  * time     (time) datetime64[ns] 176B 1998-01-01 1998-02-01 ... 2005-01-01
Attributes:
    units:          mg m^-3
    standard_name:  mass_concentration_of_chlorophyll_in_sea_water
    long_name:      Concentration of chlorophyll-a Mean
    valid_minimum:  0.0001
    valid_maximum:  100.0
    cell_methods:   time: mean

### Process Dataset (special cases)
* Phytoplankton Size Class
* Fronts (TBD)

In [22]:
xr.open_mfdataset(get_prod_files('PSC'))

📦 Found 218 .nc files in: /Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/PRODUCTS/NES_4KM_DAILY/PSC


<xarray.Dataset> Size: 434MB
Dimensions:     (time: 218, lat: 288, lon: 432)
Coordinates:
  * lat         (lat) float64 2kB 45.98 45.94 45.9 45.85 ... 34.1 34.06 34.02
  * lon         (lon) float64 3kB -78.98 -78.94 -78.9 ... -61.1 -61.06 -61.02
  * time        (time) datetime64[ns] 2kB 1998-01-01 1998-01-02 ... 2005-01-09
Data variables:
    PSC_fmicro  (time, lat, lon) float32 108MB dask.array<chunksize=(1, 288, 432), meta=np.ndarray>
    PSC_fnano   (time, lat, lon) float32 108MB dask.array<chunksize=(1, 288, 432), meta=np.ndarray>
    PSC_fpico   (time, lat, lon) float32 108MB dask.array<chunksize=(1, 288, 432), meta=np.ndarray>
    CHL         (time, lat, lon) float32 108MB dask.array<chunksize=(1, 288, 432), meta=np.ndarray>
Attributes: (12/59)
    conventions:                   CF-1.11, COARDS, ACDD-1.3
    acknowledgement:               The data are sponsored by NOAA and may be ...
    institution:                   DOC | NOAA | National Marine Fisheries Ser...
    naming_authority:              gov.noaa.nefsc
    license:                       The data may be used and redistributed for...
    creator_type:                  group
    ...                            ...
    source_sst_agency:             National Oceanic and Atmospheric Administr...
    source_sst_program:            NOAA Coral Reef Watch Program
    source_sst_dataset_url:        https://coralreefwatch.noaa.gov/main/
    source_sst_dataset_doi:        doi: 10.25921/6jgr-pt28
    source_sst_dataset_reference:  Liu, Gang; Heron, Scott F.; Eakin, C. Mark...
    source_sst_version_url:        https://www.ncei.noaa.gov/access/metadata/...

In [2]:
xr.open_mfdataset(get_prod_files('PSC',period='M',map_region='NES'))

📦 Found 13 .nc files in: /Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/PRODUCTS/NES_4KM_MONTHLY/PSC


<xarray.Dataset> Size: 408MB
Dimensions:            (time: 13, lat: 288, lon: 432)
Coordinates:
  * lat                (lat) float64 2kB 45.98 45.94 45.9 ... 34.1 34.06 34.02
  * lon                (lon) float64 3kB -78.98 -78.94 -78.9 ... -61.06 -61.02
  * time               (time) datetime64[ns] 104B 1998-01-01 ... 2005-01-01
Data variables: (12/56)
    PSC_fmicro_mean    (time, lat, lon) float32 6MB dask.array<chunksize=(1, 288, 432), meta=np.ndarray>
    PSC_fmicro_min     (time, lat, lon) float32 6MB dask.array<chunksize=(1, 288, 432), meta=np.ndarray>
    PSC_fmicro_max     (time, lat, lon) float32 6MB dask.array<chunksize=(1, 288, 432), meta=np.ndarray>
    PSC_fmicro_median  (time, lat, lon) float32 6MB dask.array<chunksize=(1, 288, 432), meta=np.ndarray>
    PSC_fmicro_std     (time, lat, lon) float32 6MB dask.array<chunksize=(1, 288, 432), meta=np.ndarray>
    PSC_fmicro_sum     (time, lat, lon) float32 6MB dask.array<chunksize=(1, 288, 432), meta=np.ndarray>
    ...                 ...
    CHL_max            (time, lat, lon) float32 6MB dask.array<chunksize=(1, 288, 432), meta=np.ndarray>
    CHL_median         (time, lat, lon) float32 6MB dask.array<chunksize=(1, 288, 432), meta=np.ndarray>
    CHL_std            (time, lat, lon) float32 6MB dask.array<chunksize=(1, 288, 432), meta=np.ndarray>
    CHL_sum            (time, lat, lon) float32 6MB dask.array<chunksize=(1, 288, 432), meta=np.ndarray>
    CHL_var            (time, lat, lon) float32 6MB dask.array<chunksize=(1, 288, 432), meta=np.ndarray>
    CHL_count          (time, lat, lon) float64 13MB dask.array<chunksize=(1, 288, 432), meta=np.ndarray>
Attributes: (12/51)
    conventions:                   CF-1.11, COARDS, ACDD-1.3
    acknowledgement:               The data are sponsored by NOAA and may be ...
    institution:                   DOC | NOAA | National Marine Fisheries Ser...
    naming_authority:              gov.noaa.nefsc
    license:                       The data may be used and redistributed for...
    creator_type:                  group
    ...                            ...
    source_dataset_url:            https://climate.esa.int/en/projects/ocean-...
    source_dataset_reference:      Sathyendranath, S, Brewin, RJW, Brockmann,...
    source_version_url:            https://catalogue.ceda.ac.uk/uuid/9aba59ec...
    source_version_doi:            doi:10.5285/5011d22aae5a4671b0cbc7d05c56c4f0
    source_version_reference:      Sathyendranath, S.; Jackson, T.; Brockmann...
    source_version:                V6.0

### Climatologies

In [6]:
files = get_prod_files('SST',period='MONTH',map_region='NES')
xr.open_mfdataset(files)

📦 Found 9 .nc files in: /Users/kimberly.hyde/Documents/nadata/DATASETS/ACSPO/V2.81/PRODUCTS/NES_2KM_CLIMATOLOGY/SST


<xarray.Dataset> Size: 175MB
Dimensions:     (time: 9, lat: 600, lon: 900)
Coordinates:
  * lat         (lat) float32 2kB 45.99 45.97 45.95 45.93 ... 34.05 34.03 34.01
  * lon         (lon) float32 4kB -78.99 -78.97 -78.95 ... -61.05 -61.03 -61.01
  * time        (time) datetime64[ns] 72B 2001-01-01T12:00:00 ... 2001-09-01T...
Data variables:
    SST_mean    (time, lat, lon) float32 19MB dask.array<chunksize=(1, 600, 900), meta=np.ndarray>
    SST_min     (time, lat, lon) float32 19MB dask.array<chunksize=(1, 600, 900), meta=np.ndarray>
    SST_max     (time, lat, lon) float32 19MB dask.array<chunksize=(1, 600, 900), meta=np.ndarray>
    SST_median  (time, lat, lon) float32 19MB dask.array<chunksize=(1, 600, 900), meta=np.ndarray>
    SST_std     (time, lat, lon) float32 19MB dask.array<chunksize=(1, 600, 900), meta=np.ndarray>
    SST_sum     (time, lat, lon) float32 19MB dask.array<chunksize=(1, 600, 900), meta=np.ndarray>
    SST_var     (time, lat, lon) float32 19MB dask.array<chunksize=(1, 600, 900), meta=np.ndarray>
    SST_count   (time, lat, lon) float64 39MB dask.array<chunksize=(1, 600, 900), meta=np.ndarray>
Attributes: (12/43)
    conventions:                   CF-1.11, COARDS, ACDD-1.3
    acknowledgement:               The data are sponsored by NOAA and may be ...
    institution:                   DOC | NOAA | National Marine Fisheries Ser...
    naming_authority:              gov.noaa.nefsc
    license:                       The data may be used and redistributed for...
    creator_type:                  group
    ...                            ...
    source_title:                  NOAA Advanced Clear-Sky Processor for Ocea...
    source_agency:                 National Oceanic and Atmospheric Administr...
    source_program:                NOAA CoastWatch
    source_dataset_url:            https://coastwatch.noaa.gov/cwn/products/a...
    source_dataset_reference:      Jonasson, O., Gladkova, I., Ignatov, A., 2...
    source_version:                V2.81

In [ ]:
files = get_prod_files('CHL',period='M',map_region='NES')
get_period_sets('MONTH', files=files)

In [5]:
get_prod_files('SST')

📦 Found 226 .nc files in: /Users/kimberly.hyde/Documents/nadata/DATASETS/ACSPO/V2.81/SOURCE/GLOBAL_2KM_DAILY/SST


['/Users/kimberly.hyde/Documents/nadata/DATASETS/ACSPO/V2.81/SOURCE/GLOBAL_2KM_DAILY/SST/20010101120000-STAR-L3S_GHRSST-SSTsubskin-LEO_Daily-ACSPO_V2.81-v02.0-fv01.0.nc',
 '/Users/kimberly.hyde/Documents/nadata/DATASETS/ACSPO/V2.81/SOURCE/GLOBAL_2KM_DAILY/SST/20010102120000-STAR-L3S_GHRSST-SSTsubskin-LEO_Daily-ACSPO_V2.81-v02.0-fv01.0.nc',
 '/Users/kimberly.hyde/Documents/nadata/DATASETS/ACSPO/V2.81/SOURCE/GLOBAL_2KM_DAILY/SST/20010103120000-STAR-L3S_GHRSST-SSTsubskin-LEO_Daily-ACSPO_V2.81-v02.0-fv01.0.nc',
 '/Users/kimberly.hyde/Documents/nadata/DATASETS/ACSPO/V2.81/SOURCE/GLOBAL_2KM_DAILY/SST/20010104120000-STAR-L3S_GHRSST-SSTsubskin-LEO_Daily-ACSPO_V2.81-v02.0-fv01.0.nc',
 '/Users/kimberly.hyde/Documents/nadata/DATASETS/ACSPO/V2.81/SOURCE/GLOBAL_2KM_DAILY/SST/20010105120000-STAR-L3S_GHRSST-SSTsubskin-LEO_Daily-ACSPO_V2.81-v02.0-fv01.0.nc',
 '/Users/kimberly.hyde/Documents/nadata/DATASETS/ACSPO/V2.81/SOURCE/GLOBAL_2KM_DAILY/SST/20010106120000-STAR-L3S_GHRSST-SSTsubskin-LEO_Daily-ACSP

In [2]:
run_stats_pipeline("PSC",periods="M",verbose=True,debug=True,subset='NES',max_workers=1)

🔍 Debug Mode: RuntimeWarnings are visible.

🚀 Starting pipeline for PSC | Period: M
🔍 Resolving output directory for PSC (M)...
📦 Selected map by data_type → NES_4KM_STATS
📂 Transformed path to MONTHLY: /Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/PRODUCTS/NES_4KM_MONTHLY/PSC
🛠 Created directory: /Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/PRODUCTS/NES_4KM_MONTHLY/PSC
🔍 DEBUG: Output directory for PSC (M) is /Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/PRODUCTS/NES_4KM_MONTHLY/PSC
Searching for D files for None:PSC
📦 Defaulted to base DAILY map → NES_4KM_DAILY
🔍 Checking PRODUCTS/GLOBAL_4KM_STATS for product 'PSC'
🔍 Checking PRODUCTS/NES_4KM_DAILY for product 'PSC'
🔍 Checking PRODUCTS/NES_4KM_MONTHLY for product 'PSC'
🔍 Checking PRODUCTS/GLOBAL_4KM_DAILY for product 'PSC'
🔍 Checking PRODUCTS/NES_4KM_CLIMS for product 'PSC'
🔍 Checking PRODUCTS/NES_4KM_STATS for product 'PSC'
🔍 Checking SOURCE/GLOBAL_4KM_DAILY for product 'PSC'
✅ Found 'PSC' in 'PR

In [7]:
ds = xr.open_dataset('/Users/kimberly.hyde/Documents/nadata/DATASETS/OCCCI/V6.0/PRODUCTS/NES_4KM_MONTHLY/PSC/M_199801-OCCCI-PSC-NES-STATS.nc')
da = ds.PSC_micro_median
ds

<xarray.Dataset> Size: 31MB
Dimensions:            (time: 1, lat: 288, lon: 432)
Coordinates:
  * lat                (lat) float64 2kB 45.98 45.94 45.9 ... 34.1 34.06 34.02
  * lon                (lon) float64 3kB -78.98 -78.94 -78.9 ... -61.06 -61.02
  * time               (time) datetime64[ns] 8B 1998-01-01
Data variables: (12/56)
    PSC_fmicro_mean    (time, lat, lon) float32 498kB ...
    PSC_fmicro_min     (time, lat, lon) float32 498kB ...
    PSC_fmicro_max     (time, lat, lon) float32 498kB ...
    PSC_fmicro_median  (time, lat, lon) float32 498kB ...
    PSC_fmicro_std     (time, lat, lon) float32 498kB ...
    PSC_fmicro_sum     (time, lat, lon) float32 498kB ...
    ...                 ...
    CHL_max            (time, lat, lon) float32 498kB ...
    CHL_median         (time, lat, lon) float32 498kB ...
    CHL_std            (time, lat, lon) float32 498kB ...
    CHL_sum            (time, lat, lon) float32 498kB ...
    CHL_var            (time, lat, lon) float32 498kB ...
    CHL_count          (time, lat, lon) float64 995kB ...
Attributes: (12/51)
    conventions:                   CF-1.11, COARDS, ACDD-1.3
    acknowledgement:               The data are sponsored by NOAA and may be ...
    institution:                   DOC | NOAA | National Marine Fisheries Ser...
    naming_authority:              gov.noaa.nefsc
    license:                       The data may be used and redistributed for...
    creator_type:                  group
    ...                            ...
    source_dataset_url:            https://climate.esa.int/en/projects/ocean-...
    source_dataset_reference:      Sathyendranath, S, Brewin, RJW, Brockmann,...
    source_version_url:            https://catalogue.ceda.ac.uk/uuid/9aba59ec...
    source_version_doi:            doi:10.5285/5011d22aae5a4671b0cbc7d05c56c4f0
    source_version_reference:      Sathyendranath, S.; Jackson, T.; Brockmann...
    source_version:                V6.0

In [ ]:
files = get_prod_files('CHL')
fp = file_parser(files[0])
fp

In [ ]:
files = get_prod_files('SST')
ds = xr.open_mfdataset(files)
ds

In [ ]:
files = get_prod_files('SST',period='M',map_region='NES')
ds = xr.open_mfdataset(files)
da = ds.SST_M_mean
ds

In [ ]:
f = get_prod_files('SST',dataset='CORALSST')
ds = xr.open_mfdataset(f)
ds

In [ ]:
get_nc_prod('CORALSST','SST')

In [ ]:
# --- Quick Test Script ---
prod = "CHL"
per  = "M"  # Monthly Mean
subset = "NES"
dataset = 'OCCCI'

run_stats_pipeline(prod,periods=per,subset=subset,dataset=dataset,verbose=True,debug=False)

In [4]:
base = ['M_200009','M_200010']
extract_period_code('MONTH_02_2000_2009')

{'code': 'MONTH',
 'full_period': 'MONTH_02_2000_2009',
 'segments': ['02', '2000', '2009'],
 'years': [2000, 2009]}

In [2]:
run_stats_pipeline('SST',periods=['M'],subset='NES',verbose=True,debug=False)

🤫 Quiet Mode: RuntimeWarnings are hidden.

🚀 Starting pipeline for SST | Period: M
🔍 Resolving output directory for SST (M)...
📦 Auto-resolved dataset_map → MAPPED_2KM_DAILY
📂 Transformed path to MONTHLY: /Users/kimberly.hyde/Documents/nadata/DATASETS/ACSPO/V2.81/PRODUCTS/NES_2KM_MONTHLY/SST
🔍 DEBUG: Output directory for SST (M) is /Users/kimberly.hyde/Documents/nadata/DATASETS/ACSPO/V2.81/PRODUCTS/NES_2KM_MONTHLY/SST
Searching for D files for None:SST
📦 Auto-resolved dataset_map → MAPPED_2KM_DAILY
🔍 Checking SOURCE/MAPPED_2KM_DAILY for product 'SST'
✅ Found 'SST' in 'SOURCE' → /Users/kimberly.hyde/Documents/nadata/DATASETS/ACSPO/V2.81/SOURCE/MAPPED_2KM_DAILY/SST
🔍 Searching for .nc files in: /Users/kimberly.hyde/Documents/nadata/DATASETS/ACSPO/V2.81/SOURCE/MAPPED_2KM_DAILY/SST
🧾 Provenance log:
  • Loaded dataset defaults → version: V2.8.1, map: GLOBAL_2KM, default_product: SST
  • Using default dataset_map: GLOBAL_2KM
  • Resolved product path → /Users/kimberly.hyde/Documents/nadata/

In [ ]:
files = get_prod_files('SST')
fp = file_parser(files[0])
fp[0]['dataset']

In [ ]:
sm = build_stats_map('SST','M')
sm[0][0]


In [ ]:
# --- Quick Test Script ---
prod = "SST"
per  = "M"  # Monthly Mean
subset = "NES"

print(f"🧪 Testing Pipeline Discovery for {prod}...")

try:
    # 1. Test Directory Resolution
    out_dir = get_prod_files(prod, period=per, map=subset, getfilepath=True)
    print(f"✅ Resolved Output Dir: {out_dir}")

    # 2. Test Map Building
    # Set overwrite=True to force 'is_up_to_date' to False for testing
    s_map = build_stats_map(prod, per, subset=subset, overwrite=True, verbose=True)

    if s_map:
        first_key = list(s_map.keys())[0]
        sample = s_map[first_key]
        print(f"\n📝 Sample Task Details for {first_key}:")
        print(f"   • Output Path: {sample['output']}")
        print(f"   • Input Count: {len(sample['inputs'])} files")
        print(f"   • Needs Update: {not sample['is_up_to_date']}")
    else:
        print("❌ Test Failed: No tasks generated.")

except Exception as e:
    print(f"💥 Test Crashed: {e}")

In [ ]:
stats_map = build_stats_map("SST", "M")
#stats_map
#pm = stats_map['period_map']
#output_periods = list(stats_map["output"].keys())
#output_periods
all_outputs = [info['output'] for info in stats_map.values()]
all_outputs[:3]

#for period, info in pm.items():
#    print(period, info["input_files"])


In [ ]:
get_prod_files('SST')

In [ ]:
path = get_prod_files('SST', period='M', getfilepath=True)


In [ ]:
run_stats_pipeline("SST")

## Demo for PERIOD functions

In [ ]:
# Determine if a period code is a climatology period
from utilities import get_period_info
info = get_period_info()
periods = info.keys()
for p in periods:
    i = info.get(p)
    print(f"{p} is climatology = {i['is_climatology']}")

In [ ]:
# Find files and dates, then get period sets for Months
from utilities import get_dates,get_period_sets, get_period_dates

dates = get_dates([2020,2025])
get_period_sets('W',dates=dates)


In [ ]:
get_period_dates('M_202001')

In [ ]:

perdates = get_period_dates(sets)
perdates
#for a, pd in zip(sets, perdates):
#    print("Week:",a,pd)

In [ ]:
# Find files and dates, then get period sets for SEA
from utilities import get_period_sets
files = get_prod_files("CHL")
dates = get_file_dates(files)
start_dates = [start for start, _ in dates]

sets = get_period_sets('SEA',dates=start_dates)

for a in sets:
    print("SEA periods:",a)

In [ ]:
# Get the dates from the input periods
from utilities import get_period_dates

files = ["D_20250926",
         "DD_20250926_20251231",
         "DOY_023_2020_2025",
         "DD3_20250310_20250312",
         "D8_20250101_20250108",
         "W_202301",
         "WW_202501_202552",
         "WEEK_22_2020_2025",
         "M_202204",
         "MM_202501_202612",
         "MM3_202501_202512",
         "MONTH_11_2020_2025",
         "A_2025",
         "ANNUAL_2020_2025",
         "Y_2025",
         "YY_2020_2025",
         "YEAR_2020_2025"
         ]

#g = get_period_dates(files)
for f in files:
    print(f"{f}: {get_file_dates([f])}")

dates = get_file_dates(files)

date_format="%Y%m%d"
valid = [
        (datetime.strptime(s, date_format), datetime.strptime(e, date_format))
        for s, e in dates
        if s != "NA" and e != "NA"
    ]

min_date = min(s for s, _ in valid)
max_date = max(e for _, e in valid)
print (min_date.strftime(date_format), max_date.strftime(date_format))




In [ ]:
run_stats_pipeline(prods='SST',dataset='CORALSST',periods=['M','MONTH'])

### Daily inputs
Create stats from the daily input files
* Monthly
* Weekly
* 3-day running
* 8-day running

In [ ]:
# Get Chlorophyll Files
get_dataset_products('OCCCI')
print(dataset_defaults()['OCCCI'])
dataset_products = get_dataset_products('OCCCI')
print(dataset_products.keys())
files = get_prod_files('CHL',dataset='OCCCI')
ds = xr.open_mfdataset(files)
ds.data_vars.keys()